# Projet 6 - Initiez-vous au MLOps (partie 1/2)


**Objectif du projet**  
Prédire la probabilité de défaut de paiement d'un client (TARGET = 1)  
→ Problème de **classification binaire déséquilibrée**

**Approche MLOps visée dans cette première partie**  
- Suivi systématique des expériences avec **MLflow**  
- Comparaison de plusieurs modèles / jeux de features  
- Versionning des modèles  



Date : Janvier 2026  
Auteur : Joannes Landy

## Étape 2 - Traquez les expérimentations avec MLFlow

Objectifs:

Des runs visibles dans l’UI MLflow avec les paramètres testés et les scores obtenus.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import mlflow
import os


# configuration des paramètres de connexion et des répertoires de travail
MLFLOW_URL = "http://localhost:5000"
DATA_DIR = str(Path.home() / "data")
EXPERIMENT_NAME = "Projet 06 - OpenClassrooms v4"


In [2]:
from sklearn.model_selection import train_test_split

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier


from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score, make_scorer, accuracy_score
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.metrics import confusion_matrix, classification_report


from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.pipeline import Pipeline


import time


output_file = DATA_DIR + "/processed/application_train_processed.parquet"
df_prepossessed = pd.read_parquet(output_file)
    
# Création du jeu de test/train
random_state = 42
test_size = 0.2
target = 'TARGET'

# Split X and y
X = df_prepossessed.drop(columns=[target])
y = df_prepossessed[target]


display("X shape;", X.shape)


'X shape;'

(307511, 8)

In [3]:
from mlflow import MlflowClient
from sklearn.metrics import precision_score, recall_score, make_scorer


client = MlflowClient(tracking_uri=MLFLOW_URL)
exp = client.get_experiment_by_name(EXPERIMENT_NAME)


# Liste des modèles pour itération
models = {
    'linear_model': LogisticRegression(),
    'rf_model': RandomForestClassifier(),

}

# Liste des parametres à tester
param_grids = {
    'linear_model': {
        'classifier__random_state': [random_state],
        #'classifier__solver': ['lbfgs'],
        'classifier__C': [0.01, 0.1],
        #'classifier__max_iter': [10, 100, 100],
        'classifier__class_weight': ['balanced'],
    },

    'rf_model': {
        'classifier__random_state': [random_state],
        'classifier__class_weight': ['balanced', 'balanced_subsample'], # autre strategie
        'classifier__n_estimators': [100, 250, 500],
        'classifier__max_depth': [3, 5, 7],
        #'min_samples_split': [2, 5, 10],
        #'min_samples_leaf': [1, 2, 4]
    },

}

# reseau de neurone : MLP, plsuieur package a explorer, pytorch,

# chercher et Stocker les meilleurs modèles

def search_bestmodel(models, param_grids, X, y):
    """ Use GridSearchCV to find best parameter, return the best model"""
    best_models = {}
    for name, model in models.items():
        print(f"\nEntraînement de {name}...")
        
        numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
        categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()


        preprocessor = ColumnTransformer(transformers=[
                ("num", StandardScaler(), numeric_features),
                ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features)
                ])
        pipeline = Pipeline(steps=[
                ("preprocessor", preprocessor),
                ("classifier", model)
                ])

        def custom_score(y_true, y_pred):
            precision = precision_score(y_true, y_pred)
            recall = recall_score(y_true, y_pred)
            return precision + 10 * recall
        
        custom_scorer = make_scorer(custom_score)
        
        # GridSearchCV
        grid = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grids[name],
            scoring=custom_scorer,
            cv=5,                  # validation croisée à 5 folds
            n_jobs=-1,             # utiliser tous les cœurs
            verbose=1              # afficher la progression
        )
        
        # Ajuster sur les données d'entraînement
        grid.fit(X, y)
        
        # Sauvegarder le meilleur modèle
        best_models[name] = grid.best_estimator_

        results_df = pd.DataFrame(grid.cv_results_)
        for index, row in results_df.iterrows():
            run = client.create_run(experiment_id=exp.experiment_id)
            run_id = run.info.run_id
            client.set_tag(run_id, "model_type", name)

            client.log_metric(run_id, "f1", row['mean_test_score'])
            
            for key, value in row['params'].items():
                client.log_param(run_id, key.replace('classifier__', ''), value)
            client.set_terminated(run_id, status="FINISHED")
            
        
        print(f"Meilleurs paramètres : {grid.best_params_}")
        print(f"Meilleur score custom : {grid.best_score_:.4f}")
    
    return best_models

print('---------------')
best_models = search_bestmodel(models, param_grids, X, y)

---------------

Entraînement de linear_model...
Fitting 5 folds for each of 2 candidates, totalling 10 fits
🏃 View run secretive-stork-355 at: http://localhost:5000/#/experiments/4/runs/30677e40c61e44faa8a776e28959656e
🧪 View experiment at: http://localhost:5000/#/experiments/4
🏃 View run tasteful-horse-24 at: http://localhost:5000/#/experiments/4/runs/4bd23569f7d744da8e64cf6015d8a6d5
🧪 View experiment at: http://localhost:5000/#/experiments/4
Meilleurs paramètres : {'classifier__C': 0.01, 'classifier__class_weight': 'balanced', 'classifier__random_state': 42}
Meilleur score F1 : 6.5012

Entraînement de rf_model...
Fitting 5 folds for each of 18 candidates, totalling 90 fits
🏃 View run glamorous-hound-128 at: http://localhost:5000/#/experiments/4/runs/dec28dc8c074498797f721c8044aad6c
🧪 View experiment at: http://localhost:5000/#/experiments/4
🏃 View run awesome-ram-516 at: http://localhost:5000/#/experiments/4/runs/f85eedb40d2f4e1881b3bbd978366b10
🧪 View experiment at: http://localhos